# 1.2 · Incremental refresh

> **Needs:** the database built by notebook 1.1; LSEG Workspace signed in.
>
> **Produces:** the database and the research cache brought up to date. Optional before the research notebooks.

Prices change every trading day, companies report quarterly, and analysts' consensus is sampled monthly. The refresh
brings the database up to date with **one command**, and it is idempotent: safe to run as often as you like.

| Data | Update frequency | Source | What the refresh requests |
|---|---|---|---|
| Index constituents and listings | a few times a month | LSEG | today's lists and all additions / deletions (small) |
| Stock and index prices, dividends, splits, shares outstanding | every trading day | LSEG | from the first of the month 10 days before the newest stored price - prices for 50 stocks per request |
| Quarterly statements (LSEG) | every few weeks | LSEG | the last year (late restatements and announcements) |
| Annual statement items (LSEG): cash flows, depreciation, inventories, bank items | every few weeks | LSEG | the last year, one request per item for each tenth of the companies (~150) |
| Benchmark prices (^GSPC, SPY...) | every trading day | Yahoo | the last 30 days |
| Financial statements (SEC) | every few weeks | SEC | the full bulk file again, but only when our copy is over a month old |
| Analysts' estimates (I/B/E/S) | every month end | LSEG | nothing until a month has closed since the newest stored consensus; then that month's consensus, revisions and the last year's actuals (~400 requests, once a month) |

The same command is therefore the *daily* refresh and, once a month, automatically the *monthly* one as well.

The table describes the US. London (the FTSE 100, 250 and SmallCap) is refreshed by the same command, straight after: the same rows, with half-yearly statements in place of quarterly ones - a few hundred LSEG requests more a day. Filing text is not part of the refresh: notebook 1.3 picks up new annual reports when it is run again.

## Design of the refresh

1. **Incremental requests.** LSEG is asked for the last few weeks, not twenty years. Prices are requested as a
   cross-section (one request per 50 stocks) rather than as per-stock histories (one request each), so the daily
   refresh costs ~1,050 of LSEG's 10,000 daily requests for the US and a few hundred more for London; analysts'
   estimates add ~400 (the US's) on the first refresh after each month end. Yahoo is asked for the last month, and the SEC bulk file is downloaded at most monthly.
2. **Idempotent writes.** Writes are *change-only*: a row identical to the stored one is not rewritten, and a request
   already answered today is read back from the raw zone instead of being sent again. A rerun after a failure
   therefore costs no LSEG requests for what was already received, and running the refresh twice in a row writes
   nothing the second time.
3. **Quality gates.** After loading, the data-quality checks run. If one **fails**, the refresh raises an error
   *before* the local research cache is updated, so bad data never reaches a backtest.

Finally the refresh prunes the raw zone. The SEC republishes its 1.3 GB file monthly and can always be downloaded
again, so only the two newest copies are kept (`uv run eqr prune-raw --dry-run` lists what would be removed).
Everything LSEG returned is kept, because requesting it again costs part of the daily allowance. Back up the `data`
folder periodically (OneDrive, an external drive): it is the only copy of every vendor response we have received.

In [1]:
import pandas as pd
from IPython.display import display

from equity_research.pipelines.refresh import refresh, written
from equity_research.research.inventory import newest
from research_core.data.db.engine import get_engine

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")

engine = get_engine()
before = newest(engine)
display(before)

,what,vendor,newest
0,analysts' consensus,refinitiv,2026-10-31
1,company reports (published),refinitiv,2027-01-08
2,company reports (published),sec,2026-11-13
3,daily prices,refinitiv,2026-11-16
4,daily prices,yahoo,2026-11-16
5,dividends and splits,refinitiv,2027-05-20
6,dividends and splits,repository,2016-04-08
7,dividends and splits,yahoo,2026-09-21
8,estimate revisions,refinitiv,2026-10-31
9,index membership changes,refinitiv,2026-11-11


## Run the refresh

In [2]:
report = refresh(engine)
changes = written(report)
for name, part in report.items():
    if isinstance(part, dict) and "complete" in part:  # a source that can stop part way (LSEG's daily limit)
        print(f"{name}: finished {part['complete']} | {part['stopped'] or 'nothing left to ask'}")
display(changes.groupby("step")[["inserted", "updated"]].sum())
display(pd.DataFrame(report["checks"]).T)

C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


C:\SourceCode\QuantResearch\src\equity_research\pipelines\load_lseg.py:913: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(
C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  day_factor = day_factor.combine_first(factors(held))


... (108 outputs left out)

C:\SourceCode\QuantResearch\src\equity_research\pipelines\load_sec.py:271:FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


refinitiv: finished True | nothing left to ask
estimates: finished True | nothing left to ask
refinitiv_uk: finished True | nothing left to ask
estimates_uk: finished True | nothing left to ask


,inserted,updated
step,,
refinitiv,0,0
refinitiv_uk,0,0
sec,0,0
yahoo,0,0


,status,value
member_day_price_coverage,warn,0.9972
benchmark_day_coverage,pass,1
unexplained_price_jumps,warn,388
suspect_halvings,warn,4
members_after_last_price,warn,60
equity_listings_without_issuer,pass,0
facts_published_before_period_end,pass,0
overlapping_membership_spells,pass,0
members_of_two_indices,pass,0
members_before_first_trade,pass,0


What changed: the most recent date of each data type, before and after the refresh.

In [3]:
after = newest(engine)
display(before.merge(after, on=["what", "vendor"], how="outer", suffixes=(" before", " after")))

,what,vendor,newest before,newest after
0,analysts' consensus,refinitiv,2026-10-31,2026-10-31
1,company reports (published),refinitiv,2027-01-08,2027-01-08
2,company reports (published),sec,2026-11-13,2026-11-13
3,daily prices,refinitiv,2026-11-16,2026-11-16
4,daily prices,yahoo,2026-11-16,2026-11-16
5,dividends and splits,refinitiv,2027-05-20,2027-05-20
6,dividends and splits,repository,2016-04-08,2016-04-08
7,dividends and splits,yahoo,2026-09-21,2026-09-21
8,estimate revisions,refinitiv,2026-10-31,2026-10-31
9,index membership changes,refinitiv,2026-11-11,2026-11-11


## Idempotence test: a second run should write nothing

Running the refresh again immediately should find nothing new. Every count below should be zero.

In [4]:
again = written(refresh(engine))
display(again.groupby("step")[["inserted", "updated"]].sum())
not_benchmarks = again[again["step"] != "yahoo"]
assert not_benchmarks[["inserted", "updated"]].to_numpy().sum() == 0, "a repeat wrote data"

C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\pipelines\load_lseg.py:913:FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


... (104 outputs left out)

C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\ingest\vendors\refinitiv\normalize.py:545:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


C:\SourceCode\QuantResearch\src\equity_research\pipelines\load_sec.py:271:FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


,inserted,updated
step,,
refinitiv,0,0
refinitiv_uk,0,0
sec,0,0
yahoo,0,0


(Yahoo sometimes revises the latest day's price after the close, in which case a few benchmark rows change on the
second run. That is a genuine change, so it is written; every other count must be zero.)

## Scheduling the refresh

The same refresh is available as a command-line call:

```
uv run eqr refresh
```

On Windows, *Task Scheduler* can run it every weekday evening after the US close (for example 23:00 UK time), with
LSEG Workspace signed in:

* **Program:** `<repository>\.venv\Scripts\eqr.exe`
* **Arguments:** `refresh`
* **Start in:** `<repository>` (the folder this repository was cloned into)

Each run is logged in `ops.job_run` (start time, duration, rows written, and the error if it failed) and each quality
check in `ops.check_result`. Only one writing job runs at a time (a database lock): a second job is refused rather
than interleaving its writes with the first.

## When the refresh stops

* **LSEG's daily limit:** the report says so (`complete: False`). The other sources still run, and the next day's
  refresh resumes where LSEG stopped.
* **A failed quality check:** a `GateFailedError` names the check, for example
  `member_day_price_coverage = 0.95 (warn >= 0.999, fail >= 0.99)`, meaning 5 % of index member-days had no price.
  Inspect the latest vendor responses (`ingest.batch`), fix the cause, and run the refresh again.